# 📓 Notebook 2: DistilBERT Fine-tuning – Multi-label Toxic Comment Detection
**Nhóm Mù Công Nghệ** | Đề tài 22 | Tuần 4  
Chạy trên **Google Colab Pro** – GPU Tesla T4

## 1. Cài đặt & kiểm tra GPU

In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn -q
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Import

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, os, warnings, time
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, hamming_loss,
    roc_auc_score, average_precision_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay
)

LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
NUM_LABELS = len(LABELS)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nDevice: {DEVICE}")


## 3. Cấu hình hyperparameter

In [ ]:
CFG = {
    'model_name'   : 'distilbert-base-uncased',
    'max_len'      : 128,
    'batch_train'  : 16,
    'batch_eval'   : 32,
    'epochs'       : 3,
    'lr'           : 2e-5,
    'weight_decay' : 0.01,
    'warmup_ratio' : 0.1,
    'dropout'      : 0.1,
    'seed'         : 42,
    'val_size'     : 0.1,   # 10% train làm validation
    'test_size'    : 0.2,
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])
print("Cấu hình:", CFG)


## 4. Tải & tiền xử lý dữ liệu

In [ ]:
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'[^a-z0-9\s!?.,\'\-]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

df = pd.read_csv('train.csv')
df['clean_text'] = df['comment_text'].apply(preprocess)
print(f"Dataset: {df.shape}")

# Chia train / test (cố định)
X_temp, X_test, y_temp, y_test = train_test_split(
    df['clean_text'].values, df[LABELS].values,
    test_size=CFG['test_size'], random_state=CFG['seed']
)
# Chia train / validation
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=CFG['val_size'], random_state=CFG['seed']
)
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")


## 5. Tính class weights (xử lý mất cân bằng)

In [ ]:
# Tính pos_weight cho BCEWithLogitsLoss
# pos_weight[i] = (N - n_pos_i) / n_pos_i
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
pos_weight = torch.tensor(neg_counts / (pos_counts + 1e-6), dtype=torch.float32).to(DEVICE)

print("Class weights (pos_weight):")
for lbl, pw in zip(LABELS, pos_weight.cpu().numpy()):
    print(f"  {lbl:<15}: {pw:.2f}")


## 6. Tokenizer & Dataset

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(CFG['model_name'])

class ToxicDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : enc['input_ids'].squeeze(0),
            'attention_mask' : enc['attention_mask'].squeeze(0),
            'labels'         : torch.tensor(self.labels[idx], dtype=torch.float32)
        }

train_ds = ToxicDataset(X_train, y_train, tokenizer, CFG['max_len'])
val_ds   = ToxicDataset(X_val,   y_val,   tokenizer, CFG['max_len'])
test_ds  = ToxicDataset(X_test,  y_test,  tokenizer, CFG['max_len'])

train_loader = DataLoader(train_ds, batch_size=CFG['batch_train'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_eval'],  shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_eval'],  shuffle=False, num_workers=2, pin_memory=True)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")


## 7. Định nghĩa mô hình DistilBERT

In [ ]:
class DistilBertMultiLabel(torch.nn.Module):
    """
    DistilBERT + custom classification head cho multi-label.
    Luồng: Input → DistilBERT Encoder (6 layers) → CLS token → Dropout → Linear(768→6) → Sigmoid
    """
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        from transformers import DistilBertModel
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        self.dropout    = torch.nn.Dropout(dropout)
        self.classifier = torch.nn.Linear(768, num_labels)

    def forward(self, input_ids, attention_mask):
        # Bước 1 & 2: Tokenizer + 6 Transformer Layers → last_hidden_state [B, 128, 768]
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        # Bước 3: Lấy CLS token [B, 768]
        cls_output = outputs.last_hidden_state[:, 0, :]
        # Bước 4: Dropout + Linear
        x = self.dropout(cls_output)
        logits = self.classifier(x)   # [B, 6]
        # Bước 5: trả logits (sigmoid áp khi inference)
        return logits

model = DistilBertMultiLabel(CFG['model_name'], NUM_LABELS, CFG['dropout']).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")


## 8. Optimizer, Scheduler, Loss

In [ ]:
total_steps   = len(train_loader) * CFG['epochs']
warmup_steps  = int(total_steps * CFG['warmup_ratio'])

optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer,
    num_warmup_steps=warmup_steps, num_training_steps=total_steps)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print(f"Total steps  : {total_steps}")
print(f"Warmup steps : {warmup_steps}")


## 9. Training Loop

In [ ]:
from torch.cuda.amp import GradScaler, autocast

scaler = GradScaler()

def train_epoch(model, loader, optimizer, scheduler, criterion, scaler):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attn_mask = batch['attention_mask'].to(DEVICE)
        labels    = batch['labels'].to(DEVICE)

        optimizer.zero_grad()
        with autocast():
            logits = model(input_ids, attn_mask)
            loss   = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def eval_epoch(model, loader, criterion, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attn_mask = batch['attention_mask'].to(DEVICE)
            labels    = batch['labels'].to(DEVICE)
            with autocast():
                logits = model(input_ids, attn_mask)
                loss   = criterion(logits, labels)
            total_loss += loss.item()
            all_logits.append(torch.sigmoid(logits).cpu().numpy())
            all_labels.append(labels.cpu().numpy())
    y_proba = np.vstack(all_logits)
    y_true  = np.vstack(all_labels)
    y_pred  = (y_proba >= threshold).astype(int)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    return total_loss / len(loader), macro_f1, y_proba, y_true


In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_macro_f1': []}
best_macro_f1 = 0
os.makedirs('checkpoints', exist_ok=True)

for epoch in range(1, CFG['epochs'] + 1):
    t0 = time.time()
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, criterion, scaler)
    val_loss, val_f1, _, _ = eval_epoch(model, val_loader, criterion)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_macro_f1'].append(val_f1)
    elapsed = (time.time() - t0) / 60

    print(f"Epoch {epoch}/{CFG['epochs']} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Macro-F1: {val_f1:.4f} | Time: {elapsed:.1f}m")

    if val_f1 > best_macro_f1:
        best_macro_f1 = val_f1
        torch.save(model.state_dict(), 'checkpoints/best_model.pt')
        print(f"  ✅ Lưu checkpoint mới (Macro-F1={val_f1:.4f})")

print(f"\nBest Val Macro-F1: {best_macro_f1:.4f}")


## 10. Vẽ training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
epochs_range = range(1, CFG['epochs'] + 1)

ax1.plot(epochs_range, history['train_loss'], 'o-', label='Train Loss', color='steelblue')
ax1.plot(epochs_range, history['val_loss'],   's-', label='Val Loss',   color='tomato')
ax1.set_title('Training & Validation Loss', fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['val_macro_f1'], 'D-', color='seagreen', label='Val Macro-F1')
ax2.set_title('Validation Macro-F1', fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Macro-F1')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('distilbert_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 11. Đánh giá trên tập Test

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load('checkpoints/best_model.pt'))
_, _, y_proba, y_true = eval_epoch(model, test_loader, criterion, threshold=0.5)

# Per-label threshold (từ kết quả threshold tuning)
THRESHOLDS = {
    'toxic': 0.35, 'severe_toxic': 0.25, 'obscene': 0.38,
    'threat': 0.20, 'insult': 0.36, 'identity_hate': 0.22
}
thresh_arr = np.array([THRESHOLDS[l] for l in LABELS])
y_pred = (y_proba >= thresh_arr).astype(int)

print("=== CLASSIFICATION REPORT (DistilBERT, per-label threshold) ===")
print(classification_report(y_true, y_pred, target_names=LABELS, digits=2))

hl  = hamming_loss(y_true, y_pred)
mif = f1_score(y_true, y_pred, average='micro', zero_division=0)
maf = f1_score(y_true, y_pred, average='macro', zero_division=0)
roc = roc_auc_score(y_true, y_proba, average='macro')
prc = average_precision_score(y_true, y_proba, average='macro')

print(f"\n=== TỔNG HỢP METRIC DISTILBERT ===")
print(f"Hamming Loss    : {hl:.4f}")
print(f"ROC-AUC (macro) : {roc:.4f}")
print(f"PR-AUC  (macro) : {prc:.4f}")
print(f"Macro F1        : {maf:.4f}")
print(f"Micro F1        : {mif:.4f}")


## 12. ROC Curve & PR Curve

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve

COLORS = ['steelblue','darkorange','green','red','purple','saddlebrown']
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for i, (lbl, c) in enumerate(zip(LABELS, COLORS)):
    fpr, tpr, _ = roc_curve(y_true[:, i], y_proba[:, i])
    auc = roc_auc_score(y_true[:, i], y_proba[:, i])
    ax1.plot(fpr, tpr, color=c, lw=1.5, label=f'{lbl} (AUC={auc:.3f})')

ax1.plot([0,1],[0,1],'k--',lw=1)
ax1.set_title('ROC Curve – DistilBERT', fontweight='bold')
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

for i, (lbl, c) in enumerate(zip(LABELS, COLORS)):
    pre, rec, _ = precision_recall_curve(y_true[:, i], y_proba[:, i])
    ap = average_precision_score(y_true[:, i], y_proba[:, i])
    ax2.plot(rec, pre, color=c, lw=1.5, label=f'{lbl} (AUC={ap:.3f})')

ax2.set_title('PR Curve – DistilBERT', fontweight='bold')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('distilbert_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Confusion Matrix từng nhãn

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (lbl, ax) in enumerate(zip(LABELS, axes.flatten())):
    cm = confusion_matrix(y_true[:, i], y_pred[:, i])
    sns.heatmap(cm, annot=True, fmt='d', cmap='viridis', ax=ax,
                xticklabels=['Not', lbl], yticklabels=['Not', lbl])
    ax.set_title(f'Confusion Matrix – {lbl}', fontweight='bold')
    ax.set_xlabel('Predicted label'); ax.set_ylabel('True label')
plt.suptitle('DistilBERT – Confusion Matrix 6 nhãn', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('distilbert_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 14. Lưu kết quả

In [ ]:
os.makedirs('results', exist_ok=True)
np.save('results/distilbert_proba.npy',  y_proba)
np.save('results/distilbert_ytrue.npy',  y_true)
np.save('results/distilbert_ypred.npy',  y_pred)
print("✅ Đã lưu predictions và checkpoints")
